# BTS Digital Twin (NVS) — Round 2: train 1 scene / lần chạy

Notebook này chỉ xử lý **1 scene duy nhất mỗi lần chạy** (biến `SCENE` ở Bước 6). Có
**7 scene round 2** (xem `pipeline/common/scenes.py`): 5 scene BTS —
`HCM0421`, `HCM0539`, `HCM0540`, `HCM0644`, `HCM0674` — và 2 scene tổng quát —
`bonsai`, `chair`.

Cách dùng: đổi `SCENE` ở Bước 6 rồi Save Version, lặp lại cho cả 7 scene (có thể chạy
song song 2 version). Mỗi scene độc lập hoàn toàn.

**KHÔNG scene nào ở round 2 có ảnh ground-truth thật** (BTC chỉ cấp `test_poses.csv`,
không có `test/images/`) — nhưng notebook này VẪN ra được điểm số nội bộ (PSNR/SSIM/
LPIPS/Score) nhờ tự tách **holdout** từ chính ảnh train (`00_make_holdout_split.py`,
quy trình ở `plan.md` mục 4) — xem biến `MODE` ở Bước 6. `MODE="holdout"` dùng để SO
SÁNH cấu hình (ra Score), `MODE="final"` dùng để train bản nộp bài thật (không ra
Score vì dùng 100% ảnh train, không còn ảnh nào giữ lại làm GT).

**Trước khi chạy, cần điền:**
1. Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL` ở Bước 3, `GDRIVE_URL` ở Bước 4 (đã điền sẵn), các biến ở Bước 6
   (`SCENE`, `MODE`, `ANTIALIASING`, `DEPTH_PRIOR`, `EXPOSURE_COMP`, `ANTENNA_FOCUS`).
3. Ước lượng thời gian train (30000 iterations) mỗi scene tuỳ độ nặng — theo dõi log
   `pipeline/work/<scene>/03_train_3dgs.log`, canh đủ trong 1 session Kaggle.

**Bảo mật:** để notebook này **Private**.

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `Hướng đi.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"
GIT_BRANCH = "coordination/round1-status"  # <-- nhánh chứa toàn bộ hạ tầng Round 2 (holdout eval, port kỹ thuật, TRR) — ĐỔI nếu đã merge vào main

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA_ROUND2/<scene>/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA_ROUND2`
cũng được — cell dưới tự dò tìm thư mục `VAI_NVS_DATA_ROUND2` ở bất kỳ độ sâu nào
trong zip).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục VAI_NVS_DATA_ROUND2 ...")

In [ ]:
# Tự dò thư mục chứa các scene round 2 phẳng (HCM0421/, chair/, bonsai/...) ở bất
# kỳ đâu trong zip vừa giải nén, rồi symlink về đúng vị trí mà
# pipeline/common/scenes.py cần: /kaggle/working/Dataset/VAI_NVS_DATA_ROUND2
#
# KHÔNG bắt buộc thư mục bọc ngoài phải tên đúng "VAI_NVS_DATA_ROUND2" — chỉ cần
# TÌM ĐƯỢC 1 thư mục (kể cả chính gốc giải nén, nếu zip không có lớp bọc ngoài)
# chứa đủ NHIỀU scene mong đợi trực tiếp bên trong. Bản cũ bắt buộc đúng tên thư
# mục nên sẽ báo lỗi "Không tìm thấy..." nếu file zip của bạn giải nén ra không có
# đúng lớp thư mục tên "VAI_NVS_DATA_ROUND2" đó (vd giải nén thẳng ra HCM0421/ ở
# gốc, hoặc thư mục bọc ngoài đặt tên khác) — dù dữ liệu vẫn đầy đủ.
#
# Danh sách tên scene lặp lại thủ công ở đây (không import common.scenes) vì
# sys.path chưa trỏ tới pipeline/ ở bước này (việc đó làm ở cell kiểm tra ngay
# sau) — giữ đồng bộ với BTS_SCENES/GENERIC_SCENES trong pipeline/common/scenes.py
# nếu sau này thêm/bớt scene.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4  # đủ scene trùng khớp để tin đây đúng là thư mục dataset (tránh khớp nhầm thư mục rác)

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (HCM0421/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/..." trước bản thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break  # khớp đủ cả 7 — dừng sớm, khỏi walk tiếp cho nhanh

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        "Không tìm thấy thư mục nào chứa đủ scene round 2 (HCM0421, chair, bonsai...) "
        f"trong zip vừa giải nén (khớp nhiều nhất: {best_match}/7, cần >= {_MIN_MATCH}).\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — chạy `!find {RAW_ROOT} -maxdepth 3` ở 1 cell "
        "khác để xem cấu trúc thật, đối chiếu lại với file zip đã upload lên Google Drive."
    )

print(f"Tìm thấy ({best_match}/7 scene khớp):", found)
target = Path("/kaggle/working/Dataset/VAI_NVS_DATA_ROUND2")
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 7 scene round 2 + scene nào có sparse hợp lệ — dataset
# đầy đủ thì kỳ vọng has_valid_provided_sparse=True cho CẢ 7 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa VAI_NVS_DATA_ROUND2 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.domain:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 (tuỳ chọn) — Sanity-check hệ toạ độ

Script `02_validate_frame.py` tự chạy lại COLMAP trên `HCM0421` (scene round-2 đã test
kỹ ở Phase 0) rồi so với sparse chính thức — round 2: cả 7/7 scene đều có sparse hợp lệ
do BTC cấp (đã xác nhận bằng test thật), nên bước này chỉ còn là đối chiếu/kiểm tra
thêm cho chắc, không bắt buộc.

In [ ]:
RUN_SANITY_CHECK = False  # <-- đổi thành True khi muốn chạy lại bước đối chiếu này

if RUN_SANITY_CHECK:
    !python /kaggle/working/pipeline/scripts/02_validate_frame.py
else:
    print("Bỏ qua sanity-check (RUN_SANITY_CHECK = False).")

## Bước 6 — Cấu hình + train 1 scene

**Đổi các biến ở cell dưới rồi Save Version** (mỗi lần 1 tổ hợp scene+config, ở version khác nhau):

- `SCENE`: 1 trong 7 tên — `HCM0421`, `HCM0539`, `HCM0540`, `HCM0644`, `HCM0674`, `bonsai`, `chair`.
- `MODE`:
  - `"holdout"` — train nhanh trên ~85-90% ảnh train (giữ lại phần còn lại làm GT nội
    bộ), render + chấm điểm (Score) trên phần giữ lại đó. **Dùng để SO SÁNH các cấu
    hình A/B/C cho từng scene — checkpoint ra từ chế độ này TUYỆT ĐỐI KHÔNG được dùng
    để nộp bài** (chỉ train trên 1 phần dữ liệu).
  - `"final"` — train trên **100% ảnh train**, render `test_poses.csv` thật. Chỉ chạy
    chế độ này SAU KHI đã chọn xong cấu hình thắng cho scene đó bằng vài lần
    `MODE="holdout"` ở trên. Đây là checkpoint dùng để nộp bài.
- `ANTIALIASING` (0/1, mặc định 1): cấu hình A (mip-splatting antialiasing-only,
  `plan.md` mục 6.1).
- `DEPTH_PRIOR` (0/1, mặc định 0): bật thêm cùng `ANTIALIASING=1` = cấu hình B (A +
  depth prior). Cần cài Depth-Anything-V2 (cell tự làm bên dưới nếu bật).
- `EXPOSURE_COMP` (0/1, mặc định 0): bật thêm nếu quan sát rõ lệch màu ở 1 scene cụ thể.
- `ANTENNA_FOCUS` (0/1, mặc định 0): **chỉ có tác dụng với 5 scene BTS** (Phase D,
  `plan.md` mục 6.1 — cần chạy `07_build_antenna_weights.py` trước để tạo
  `antenna_weights.json`, xem docstring file đó). Với `bonsai`/`chair`, cờ này bị bỏ
  qua dù bật (script tự chặn).

Chi tiết đầy đủ ghi ra file `pipeline/work/<scene>/03_train_3dgs.log`. Xem tiến độ lúc
train đang chạy: mở 1 cell khác gõ `!tail -n 30 /kaggle/working/pipeline/work/<scene>/03_train_3dgs.log`.

In [ ]:
SCENE = "HCM0421"  # <-- đổi thành tên scene muốn train ở version này
MODE = "holdout"   # "holdout" (so Score config, KHÔNG dùng nộp bài) hoặc "final" (100% data, dùng để nộp)

ANTIALIASING = 1   # 0/1 — cấu hình A (mip-splatting antialiasing-only)
DEPTH_PRIOR = 0    # 0/1 — bật thêm = cấu hình B (A + depth prior), cần Depth-Anything-V2 (cell dưới)
EXPOSURE_COMP = 0  # 0/1 — chỉ bật nếu quan sát rõ lệch màu ở scene cụ thể
ANTENNA_FOCUS = 0  # 0/1 — chỉ có tác dụng với 5 scene BTS, cần antenna_weights.json (07_build_antenna_weights.py) trước

assert MODE in ("holdout", "final"), 'MODE phải là "holdout" hoặc "final"'

# In BANNER dễ thấy — đã từng có lần chạy thật quên đổi DEPTH_PRIOR trước khi
# chạy cả notebook (~1 tiếng GPU Kaggle), ra nhầm dữ liệu cấu hình A dán nhầm
# tên thư mục cấu hình B. Đọc kỹ dòng dưới TRƯỚC KHI chạy tiếp các cell sau.
print("=" * 78)
print(f"  SCENE={SCENE}  MODE={MODE}")
print(f"  ANTIALIASING={ANTIALIASING}  DEPTH_PRIOR={DEPTH_PRIOR}  EXPOSURE_COMP={EXPOSURE_COMP}  "
      f"ANTENNA_FOCUS={ANTENNA_FOCUS}")
print("  --> KIỂM TRA LẠI ĐÚNG CẤU HÌNH ĐỊNH CHẠY TRƯỚC KHI CHẠY CÁC CELL TIẾP THEO! <--")
print("=" * 78)


In [ ]:
import os

if MODE == "holdout":
    # Tạo holdout nếu chưa có (script tự báo lỗi rõ ràng + gợi ý --overwrite nếu đã
    # tồn tại — không phải lỗi, version trước có thể đã tạo rồi, bỏ qua an toàn).
    holdout_dir = f"/kaggle/working/pipeline/work/{SCENE}/holdout"
    if not os.path.isdir(holdout_dir):
        !python /kaggle/working/pipeline/scripts/00_make_holdout_split.py --scene {SCENE}
    else:
        print(f"Đã có {holdout_dir} — bỏ qua tạo lại (dùng --overwrite thủ công nếu muốn tạo lại).")
    !python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE} --holdout
else:
    !python /kaggle/working/pipeline/scripts/01_run_colmap.py --scene {SCENE}

### (Chỉ khi `DEPTH_PRIOR = 1`) Cài Depth-Anything-V2 + sinh depth prior

Cell dưới tự bỏ qua nếu `DEPTH_PRIOR = 0`. Nếu bật, cell này clone
`Depth-Anything-V2`, pin đúng commit, tải checkpoint `vitl`, rồi chạy
`08_generate_depth_priors.py` — PHẢI chạy sau bước COLMAP ở trên (cần
`colmap/dense/` tồn tại) và TRƯỚC bước train bên dưới.

In [ ]:
if DEPTH_PRIOR:
    %cd /kaggle/working
    !git clone https://github.com/DepthAnything/Depth-Anything-V2.git
    %cd /kaggle/working/Depth-Anything-V2
    !git checkout a561b849ebae10a6f5ef49e26c83cbbcd36c71bf
    !pip install -q -r requirements.txt
    !mkdir -p checkpoints
    !wget -q -O checkpoints/depth_anything_v2_vitl.pth \
        https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth

    import os
    os.environ["DA_REPO"] = "/kaggle/working/Depth-Anything-V2"
    print("DA_REPO =", os.environ["DA_REPO"])

    %cd /kaggle/working
    !python /kaggle/working/pipeline/scripts/08_generate_depth_priors.py --scene {SCENE}
else:
    print("DEPTH_PRIOR = 0 — bỏ qua cài Depth-Anything-V2/sinh depth prior.")

In [ ]:
import os

# holdout: iteration ít hơn (đủ tín hiệu so sánh tương đối giữa cấu hình, tiết
# kiệm quota GPU Kaggle vì chạy nhiều lần/scene) — final: 30000 (mặc định repo,
# chất lượng cao nhất cho bản nộp thật, chỉ chạy 1 lần/scene).
os.environ["ITERATIONS"] = "15000" if MODE == "holdout" else "30000"
os.environ["ANTIALIASING"] = str(ANTIALIASING)
os.environ["DEPTH_PRIOR"] = str(DEPTH_PRIOR)
os.environ["EXPOSURE_COMP"] = str(EXPOSURE_COMP)
os.environ["ANTENNA_FOCUS"] = str(ANTENNA_FOCUS)

# CUDA OOM THẬT đã gặp 6/6 scene khi DEPTH_PRIOR=1 (GPU Kaggle ~14.56GiB) —
# tràn bộ nhớ giữa chừng (59%-97% số iteration đã chạy), không phải do 1 scene
# cụ thể mà do densify sinh thêm Gaussian tích luỹ dần tới khi tràn, depth prior
# đẩy nó qua ngưỡng. Áp đúng tổ hợp biện pháp script đã khuyến nghị sẵn (xem
# comment đầu 03_train_3dgs.sh) CHỈ khi DEPTH_PRIOR=1 — cấu hình A (không depth)
# đã chạy trót lọt 7/7 ở mặc định nên không đụng vào.
# LƯU Ý: Score cấu hình B từ đây không còn tách bạch 100% do riêng depth prior —
# là depth_prior + SH_DEGREE=2 + densify_grad_threshold cao hơn cộng lại. Nếu
# muốn so sánh tinh khiết hơn, cần thử lại không giảm hoặc train trên GPU lớn hơn.
if DEPTH_PRIOR:
    os.environ["SH_DEGREE"] = "2"
    os.environ["DENSIFY_GRAD_THRESHOLD"] = "0.0004"
    print("DEPTH_PRIOR=1 -> tự giảm SH_DEGREE=2, DENSIFY_GRAD_THRESHOLD=0.0004 để tránh "
          "CUDA OOM (đã gặp 6/6 scene ở mặc định).")

print("ITERATIONS =", os.environ["ITERATIONS"])

!bash /kaggle/working/pipeline/scripts/03_train_3dgs.sh {SCENE}

In [ ]:
if MODE == "holdout":
    poses_csv = f"/kaggle/working/pipeline/work/{SCENE}/holdout/holdout_poses.csv"
    out_dir = f"/kaggle/working/pipeline/work/{SCENE}/holdout_renders"
    !python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {SCENE} \
        --poses_csv {poses_csv} --out_dir {out_dir}
else:
    !python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {SCENE}

In [ ]:
if MODE == "holdout":
    !python /kaggle/working/pipeline/scripts/05_eval_metrics.py --scene {SCENE}
else:
    print("MODE=final — không có ảnh GT để chấm điểm (dùng 100% ảnh train). "
          "Điểm tham khảo cho cấu hình này lấy từ lần chạy MODE=holdout trước đó.")

### Gợi ý — sanity-check train/render nhất quán

Nên chạy **1 lần** sau lần train ĐẦU TIÊN của MỖI cấu hình mới (vd lần đầu bật
`DEPTH_PRIOR=1`, hoặc lần đầu đổi `ANTIALIASING`) để bắt sớm lỗi lệch cấu hình lúc
train vs lúc render (đã từng xảy ra thật ở round 1 — xem docstring đầu
`10_sanity_check_render.py`). Không bắt buộc chạy lại mỗi version.

In [ ]:
!python /kaggle/working/pipeline/scripts/10_sanity_check_render.py --scene {SCENE}

### Xem thử vài ảnh render ra (kiểm tra bằng mắt)

Kiểm tra hợp lý (không nhiễu loạn, không sai màu/hình dạng bất thường) trước khi lưu
lên Drive. Nếu `MODE="holdout"`, Score định lượng đã in ở cell trên rồi — đây chỉ là
kiểm tra bằng mắt bổ sung.

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

renders_dir = Path(f"/kaggle/working/pipeline/work/{SCENE}/holdout_renders" if MODE == "holdout"
                    else f"/kaggle/working/pipeline/work/{SCENE}/renders")
all_renders = sorted(renders_dir.glob("*.png"))
sample = all_renders[:4]
print(f"{len(all_renders)} ảnh render tại {renders_dir}, xem thử {len(sample)} ảnh đầu:")
for p in sample:
    display(Image.open(p))

## Bước 7 — Lấy checkpoint để lưu lên Google Drive

**Chỉ làm bước này khi `MODE = "final"`.** Nếu vừa chạy `MODE = "holdout"`, checkpoint
vừa train chỉ dùng để đo Score so sánh cấu hình — **KHÔNG tải lên Drive / KHÔNG dùng để
nộp bài** (chỉ train trên ~85-90% ảnh train, thiếu dữ liệu so với bản nộp thật).

Quy trình đầy đủ cho MỖI scene:
1. Chạy vài version với `MODE="holdout"`, đổi `ANTIALIASING`/`DEPTH_PRIOR`/
   `EXPOSURE_COMP` để so Score (cell "Bước 6" ở trên) — chọn cấu hình có Score cao nhất
   cho scene đó (không quyết định bằng mắt, xem `plan.md` mục 4).
2. Chạy 1 version cuối với `MODE="final"` + đúng cấu hình đã thắng — đây mới là
   checkpoint dùng để nộp bài, làm theo hướng dẫn lấy checkpoint bên dưới.

Không cần nén gì — tải thẳng cả thư mục rồi upload nguyên vậy lên Drive.

Checkpoint (trọng số đã train) nằm ở:
`pipeline/work/<SCENE>/gs_model/point_cloud/iteration_30000/point_cloud.ply`
(kèm 2 checkpoint giữa chừng ở `iteration_7000/` và `iteration_15000/`, phòng khi cần
iteration cuối bị lỗi).

Cách lấy: bấm **Save Version**, vào tab **Output**, tìm đúng thư mục
`pipeline/work/<SCENE>/gs_model/` (không cần lấy nguyên `/kaggle/working` — phần còn lại
chỉ là code/dataset/repo clone, không cần cho submission), tải thư mục này về máy rồi
upload thẳng lên Google Drive dưới dạng 1 thư mục — đặt tên thư mục trên Drive rõ theo
tên scene (vd `<SCENE>_gs_model`) để không nhầm lẫn khi điền link ở
`kaggle_submission.ipynb`. Nhớ đổi chế độ share thư mục đó thành "Anyone with the link".

Lặp lại (holdout x N lần so cấu hình + final x 1 lần) cho cả 7 scene round 2.